# local_polynomial

> 对应代码：`EconometricsCode/code_in_notes/Chap.3/local_polynomial.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.3`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.3")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


文件开始，同样先清空内存、关闭分屏显示，保证环境干净、输出连续。

In [2]:
%%stata
clear
set more off


. clear

. set more off

. 


我们使用与 local_linear.do 完全相同的模拟数据（相同种子、相同 DGP：x~U(0,2)，m(x)=exp(sin(x³))，加标准正态噪声），以便在同一批数据上比较不同阶数的局部多项式估计。

In [3]:
%%stata
// 生成数据
set seed 19880505
set obs 300
gen x=2*runiform()
gen y=exp(sin(x^3))+rnormal()


. // 生成数据
. set seed 19880505

. set obs 300
Number of observations (_N) was 0, now 300.

. gen x=2*runiform()

. gen y=exp(sin(x^3))+rnormal()

. 


构造与之前相同的核权重：以 x₀=2 为中心、带宽 h=0.1 的正态核，离 2 越近的观测权重越大（此处注释沿用了上一文件的写法，实际上我们要做的是更高阶的局部多项式回归）。

In [4]:
%%stata
// 局部线性回归
gen w=normalden((x-2)/0.1)


. // 局部线性回归
. gen w=normalden((x-2)/0.1)

. 


在一阶项 x_2=x-2 的基础上，我们再生成二阶项 x_22=x_2² 和三阶项 x_23=x_2³，把局部逼近从一条直线升级为三次多项式。

In [5]:
%%stata
gen x_2=x-2
gen x_22=x_2^2
gen x_23=x_2^3


. gen x_2=x-2

. gen x_22=x_2^2

. gen x_23=x_2^3

. 


核心估计命令：通配符 x_2* 同时选中 x_2、x_22、x_23，配合 [iw=w] 做加权最小二乘，即在 x₀=2 处拟合阶数 p=3 的局部多项式：最小化 Σᵢ wᵢ(yᵢ−β₀−β₁(xᵢ−2)−β₂(xᵢ−2)²−β₃(xᵢ−2)³)²。提高多项式阶数可以更灵活地逼近真实函数的曲率，通常能降低偏差，但以增加方差为代价；实践中常取 p=1（局部线性）或 p=2（局部二次）。

In [6]:
%%stata
reg y x_2* [iw=w]


      Source |       SS           df       MS      Number of obs   =         8
-------------+----------------------------------   F(3, 4)         =      0.57
       Model |  3.00097743         3  1.00032581   Prob > F        =    0.6638
    Residual |  7.30412813         4  1.82603203   R-squared       =    0.2912
-------------+----------------------------------   Adj R-squared   =   -0.1915
       Total |  10.3051056         7  1.47215794   Root MSE        =    1.3244

------------------------------------------------------------------------------
           y | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
         x_2 |  -5.460779   46.31851    -0.12   0.912    -134.0616      123.14
        x_22 |  -163.3487   418.3276    -0.39   0.716    -1324.812     998.115
        x_23 |  -416.6826   989.4088    -0.42   0.695    -3163.722    2330.357
       _cons |   2.495374   1.240511     2.01   0.

同样显示常数项，即局部三次多项式给出的 m̂(2)，可与真实值 exp(sin(8))≈2.69 以及局部线性的结果对照，直观感受提高阶数对估计的影响。

In [7]:
%%stata
di _b[_cons]

2.4953737
